In [0]:
# Installs OpenAI SDK, Pydantic, NetworkX (for Tier 3 graph traversal), and Matplotlib
%pip install -q openai pydantic networkx matplotlib
%restart_python

In [0]:
# Creates a secure text box at the top of your notebook UI so your API key is NEVER saved to GitHub
dbutils.widgets.text("llm_api_key", "", "Paste API Key Here")
print("Look at the top of your notebook and paste your OpenAI or Gemini API key into 'Paste API Key Here'.")


In [0]:
import sys
sys.path.append("../src")

from agents import ThreeTierBenchmarkAgents

API_KEY = dbutils.widgets.get("llm_api_key")
assert API_KEY != "", "Please paste your API key into the 'Paste API Key Here' widget at the top!"

# Configure for OpenAI (default) OR uncomment the two Gemini lines below if using a Google AI Studio key:

# MODEL_NAME = "gemini-2.5-flash"
# BASE_URL = "https://googleapis.com"

MODEL_NAME = "gemini-4-hydrogen"
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

agents = ThreeTierBenchmarkAgents(spark=spark, api_key=API_KEY, model=MODEL_NAME, base_url=BASE_URL)

# Test 1 multi-hop causal question across all 3 tiers
test_q = "Which competitor brand do shoppers most frequently switch to when abandoning samsung?"

for name, runner in [
    ("Tier 1 (Raw Logs)", agents.run_tier1_raw_logs),
    ("Tier 2 (Semantic Layer)", agents.run_tier2_semantic_layer),
    ("Tier 3 (Ontology)", agents.run_tier3_ontology),
]:
    res = runner("TEST_01", test_q)
    print(f"=== {name} ===")
    print(f"Entity: {res.entity_answer} | Numeric: {res.numeric_answer} | Tokens: {res.prompt_tokens + res.completion_tokens} | Latency: {res.latency_sec}s")
    print(f"Code/Tool: {res.generated_code_or_tool}")
    if res.execution_error:
        print(f"Error: {res.execution_error}")
    print()
